In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.plot_helpers import make_histogram_plot
from src.ntuple_variables.variables import combined_training_vars
from src.df_helpers import lazy_height


In [ ]:
# after running python src/create_splines_df.py -f 0.05

In [ ]:
training = "all_vars"


In [ ]:
# Load the normal df (lazy)
print("Loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df_with_splines.parquet")
print(f"Total events: {lazy_height(all_df)}")

# Load the BDT predictions df (lazy)
print("Loading predictions.parquet...")
predictions_df = pl.scan_parquet(f"../training_outputs/{training}/predictions.parquet")
print(f"Total events: {lazy_height(predictions_df)}")

# Load the spline df (lazy)
print("Loading spline_weights_df.parquet...")
spline_weights_df = pl.scan_parquet(f"{intermediate_files_location}/all_df_with_splines.parquet")
print(f"Total events: {lazy_height(spline_weights_df)}")



In [ ]:
all_df = all_df.filter((pl.col("filetype") == "nu_overlay") & (pl.col("detailed_run_period") == "4b") & (pl.col("wc_kine_reco_Enu") > 0)).sort(["run", "subrun", "event"])
predictions_df = predictions_df.filter((pl.col("filetype") == "nu_overlay")).sort(["run", "subrun", "event"])
spline_weights_df = spline_weights_df.filter((pl.col("filetype") == "nu_overlay")).sort(["run", "subrun", "event"])


In [ ]:
all_df.select(["filetype", "detailed_run_period", "run", "subrun", "event", "wc_kine_reco_Enu"]).collect()

In [ ]:
predictions_df.select(["filetype", "run", "subrun", "event", "prob_1gNp"]).collect()


In [ ]:
spline_weights_df.select(["filetype", "run", "subrun", "event", "wc_kine_reco_Enu", "spline_weights_MaCCQE_UBGenie"]).collect()


In [ ]:

# Overlap analysis across the three dataframes
# Collect just the id columns from each
print("Collecting id columns...")
ids_all = all_df.select(["run", "subrun", "event"]).collect()
ids_pred = predictions_df.select(["run", "subrun", "event"]).collect()
ids_spline = spline_weights_df.select(["run", "subrun", "event"]).collect()

n_all = len(ids_all)
n_pred = len(ids_pred)
n_spline = len(ids_spline)

print(f"\nEvents per dataframe:")
print(f"  all_df:           {n_all:>8,}")
print(f"  predictions_df:   {n_pred:>8,}")
print(f"  spline_weights_df:{n_spline:>8,}")

# Pairwise: all_df vs predictions_df
in_all_and_pred = ids_all.join(ids_pred, on=["run", "subrun", "event"], how="inner")
in_all_not_pred = ids_all.join(ids_pred, on=["run", "subrun", "event"], how="anti")
in_pred_not_all = ids_pred.join(ids_all, on=["run", "subrun", "event"], how="anti")

# Pairwise: all_df vs spline_weights_df
in_all_and_spline = ids_all.join(ids_spline, on=["run", "subrun", "event"], how="inner")
in_all_not_spline = ids_all.join(ids_spline, on=["run", "subrun", "event"], how="anti")
in_spline_not_all = ids_spline.join(ids_all, on=["run", "subrun", "event"], how="anti")

# Pairwise: predictions_df vs spline_weights_df
in_pred_and_spline = ids_pred.join(ids_spline, on=["run", "subrun", "event"], how="inner")
in_pred_not_spline = ids_pred.join(ids_spline, on=["run", "subrun", "event"], how="anti")
in_spline_not_pred = ids_spline.join(ids_pred, on=["run", "subrun", "event"], how="anti")

# Three-way overlap
in_all_three = in_all_and_pred.join(ids_spline, on=["run", "subrun", "event"], how="inner")
in_all_pred_not_spline = in_all_and_pred.join(ids_spline, on=["run", "subrun", "event"], how="anti")
in_all_spline_not_pred = in_all_and_spline.join(ids_pred, on=["run", "subrun", "event"], how="anti")
in_pred_spline_not_all = in_pred_and_spline.join(ids_all, on=["run", "subrun", "event"], how="anti")
in_all_only = in_all_not_pred.join(ids_spline, on=["run", "subrun", "event"], how="anti")
in_pred_only = in_pred_not_all.join(ids_spline, on=["run", "subrun", "event"], how="anti")
in_spline_only = in_spline_not_all.join(ids_pred, on=["run", "subrun", "event"], how="anti")

print(f"\nPairwise overlaps:")
print(f"  all_df ∩ predictions_df:           {len(in_all_and_pred):>8,}  ({len(in_all_and_pred)/n_all:.1%} of all_df, {len(in_all_and_pred)/n_pred:.1%} of predictions_df)")
print(f"  all_df ∩ spline_weights_df:        {len(in_all_and_spline):>8,}  ({len(in_all_and_spline)/n_all:.1%} of all_df, {len(in_all_and_spline)/n_spline:.1%} of spline_weights_df)")
print(f"  predictions_df ∩ spline_weights_df:{len(in_pred_and_spline):>8,}  ({len(in_pred_and_spline)/n_pred:.1%} of predictions_df, {len(in_pred_and_spline)/n_spline:.1%} of spline_weights_df)")

print(f"\nThree-way breakdown:")
print(f"  in all three:                      {len(in_all_three):>8,}")
print(f"  in all_df + predictions, not spline:{len(in_all_pred_not_spline):>8,}")
print(f"  in all_df + spline, not predictions:{len(in_all_spline_not_pred):>8,}")
print(f"  in predictions + spline, not all_df:{len(in_pred_spline_not_all):>8,}")
print(f"  in all_df only:                    {len(in_all_only):>8,}")
print(f"  in predictions_df only:            {len(in_pred_only):>8,}")
print(f"  in spline_weights_df only:         {len(in_spline_only):>8,}")

print(f"\nMissing from each:")
print(f"  in all_df but not predictions_df:  {len(in_all_not_pred):>8,}  ({len(in_all_not_pred)/n_all:.1%} of all_df)")
print(f"  in predictions_df but not all_df:  {len(in_pred_not_all):>8,}  ({len(in_pred_not_all)/n_pred:.1%} of predictions_df)")
print(f"  in all_df but not spline_weights:  {len(in_all_not_spline):>8,}  ({len(in_all_not_spline)/n_all:.1%} of all_df)")
print(f"  in spline_weights but not all_df:  {len(in_spline_not_all):>8,}  ({len(in_spline_not_all)/n_spline:.1%} of spline_weights_df)")
print(f"  in predictions but not spline:     {len(in_pred_not_spline):>8,}  ({len(in_pred_not_spline)/n_pred:.1%} of predictions_df)")
print(f"  in spline but not predictions:     {len(in_spline_not_pred):>8,}  ({len(in_spline_not_pred)/n_spline:.1%} of spline_weights_df)")
